# Phylotrajectories.jl — usage example on simulated data

This notebook reproduces the full **OU‑MCMC + UMAP overlay** pipeline of
`Phylotrajectories.jl` on a small simulated dataset that ships with the
package (`examples/data/simulated_clone_data.tsv` and
`examples/data/simulated_UMAP_coords.tsv`).  The data were generated with
`sim_count_matrix` — see `examples/generate_simulated_data.jl`.

The pipeline mirrors the analysis script used for the *DogAllergen* dataset
in the paper:

1.  Read a per‑cell long‑form table → cluster names + count matrix.
2.  Run **`tree_inference`** with an Ornstein–Uhlenbeck continuous model and
    Metropolis–Hastings MCMC.
3.  Build a **HIPSTR** consensus tree from the posterior samples and plot
    it together with the MCMC cloud.
4.  Aggregate posterior samples into a **clone × node frequency matrix**
    via `run_ou_and_build_clone_matrix`.
5.  Project the consensus tree onto a **simulated UMAP** of the cells using
    Brownian motion + Felsenstein, with `PlotTreeOnUmap*` helpers.


## 1. Set up the environment

Activate the package, load the dependencies, and fix the random seed.

In [ ]:
import Pkg
# Activate the parent package so `using Phylotrajectories` resolves to this checkout.
Pkg.activate(joinpath(@__DIR__, ".."))
# Make sure all transitive deps are installed (idempotent).
# Pkg.instantiate()


In [ ]:
using Phylotrajectories
using MolecularEvolution
using ForwardBackward

using CSV, DataFrames
using Distributions, Statistics, StatsBase, LinearAlgebra
using Plots, Plots.PlotMeasures, ColorSchemes
using Phylo
using Random
using Printf

const NUMBER_SEED = 1234
Random.seed!(NUMBER_SEED);


## 2. Output directory

All figures, tree files and clone matrices produced below land in
`examples/results/`. We create it once and reuse the same `path` everywhere
— equivalent to the `path = "..."` constant in the original analysis script.

In [ ]:
path = joinpath(@__DIR__, "results")
isdir(path) || mkpath(path)
@info "Saving outputs under" path


## 3. Load the simulated count matrix

`import_count_matrix` ingests the long-form table (one row per cell) and
returns:

- `clono_info`     — DataFrame with the CDR3 sequences attached to each clone,
- `cluster_names`  — cell-type names (rows of the count matrix),
- `cluster_sizes`  — total cells per cell type,
- `count_matrix`   — `cell_types × clonotypes` integer matrix.

In [ ]:
data_dir = joinpath(@__DIR__, "data")
clono_info, cluster_names, cluster_sizes, count_matrix = import_count_matrix(
    joinpath(data_dir, "simulated_clone_data.tsv"),
    :Clonotype,
    :cell_types,
    :TRB_cdr3aa,
)

@info "loaded" cluster_names size(count_matrix) cluster_sizes
first(clono_info, 5)


## 4. Hyper‑parameters & MCMC model factory

`digamma_value`/`trigamma_value` are the pseudo‑counts that get added to the
zero entries of the count matrix before computing the digamma / trigamma
transformations used to seed the leaf Gaussian likelihoods.

`eqmu`, `eqtheta` and `v` are the initial values of the OU process
(equilibrium mean, mean‑reversion strength, and variance respectively).

In [ ]:
digamma_value  = 0.5
trigamma_value = 0.5

eqmu    = 1.5
eqtheta = 0.1
v       = 1.0

tree_warmup_cycles  = 100
start_branch_length = 0.1

function make_ou_model(;
    nni = 1,
    branchlength = 1,
    root = 1,
    models = 1,
    tree_warmup_cycles = 100,
    burn_in = 2_000,
    sample_interval = 50,
    n_samples = 200,
)
    return OUContinuousModel(
        update = OUContinuousUpdate(
            nni = nni,
            branchlength = branchlength,
            root = root,
            models = models,
            branchlength_sampler = BranchlengthSampler(Normal(0, 0.1), Normal(-1, 1)),
            root_sampler = OUGaussianStateSample(
                MvNormal(zeros(2), Diagonal([0.01, 0.01])),
                MvNormal(zeros(2), Diagonal([1.0, 0.1])),
                1e-1, 1,
            ),
            ou_eqmu_sampler = OUEqmuSampler(Normal(0.0, 2.0), Normal(1.5, 1.0), 1.0, 0.1),
            ou_theta_sampler = OUThetaSampler(Normal(0, 0.5), Normal(-1, 1), 1.5, 1.0),
        ),
        start_branch_length = start_branch_length,
        tree_warmup_cycles = tree_warmup_cycles,
        burn_in = burn_in,
        sample_interval = sample_interval,
        n_samples = n_samples,
    )
end


## 5. Run OU‑MCMC inference

> The numbers below are tuned for an **example notebook** — they finish in
> minutes, not hours. For real analyses, scale `burn_in`, `sample_interval`
> and `n_samples` up by ~10×.

In [ ]:
println("=" ^ 60)
println("Running OU MCMC inference on simulated data...")
println("=" ^ 60)

plot_init, init_tree, trees, LLs, models, root_ps, upd =
    tree_inference(
        make_ou_model(
            burn_in = 2_000,
            sample_interval = 50,
            n_samples = 200,
            tree_warmup_cycles = 100,
        ),
        cluster_names,
        count_matrix;
        eqmu = eqmu,
        eqtheta = eqtheta,
        v = v,
        d = digamma_value,
        g = trigamma_value,
    );


## 6. HIPSTR consensus + posterior cloud

`HIPSTR` builds a maximum‑credibility tree from the posterior sample
(returned by `tree_inference`).  We also keep the per‑node **log‑credibility**
and **support** values, which we'll use in the credibility plot below.

In [ ]:
ladderize!.(trees)
hip, node2logcred, node2support = HIPSTR(trees; getcred = true, getsupport = true)
mltpl = plot_multiple_trees(trees, hip; line_width = 0.075);


## 7. Diagnostic dashboard

Standard OU‑MCMC diagnostics: log‑likelihood trace, equilibrium mean
acceptance ratio, mean‑reversion `θ`, variance, and the (1‑D marginal of
the) root state distribution across samples.

In [ ]:
lls = plot(LLs, legend = :none);  title!(lls, "LLs");

means = plot([x[3] for x in models], legend = :none);
title!(means, "EqMu Acc Ratio $(round(upd.models_update.eqmu_sampler.acc_ratio[1], digits=3))");

theta = plot([x[1] for x in models], legend = :none);
title!(theta, "Theta Acc Ratio $(round(upd.models_update.theta_sampler.acc_ratio[1], digits=3))");

vars = plot([x[2] for x in models], legend = :none);
title!(vars, "Var");

plot_root = plot(xlims = (-5, 5), legend = false)
for p in root_ps
    plot!(plot_root, x -> pdf(Normal(p[1][1], p[1][2]), x))
end
title!(
    plot_root,
    "Root Acc Position: $(round(upd.root_update.acc_ratio.position[1], digits=3))" *
    " State: $(round(upd.root_update.acc_ratio.state[1], digits=3))",
);

combined_plt = plot(
    [means, vars, theta, lls, plot_root, mltpl]...;
    layout = (2, 3), size = (1500, 600),
);

hipster_tree = plot(mltpl; fontsize = 10, margins = 10mm);

savefig(combined_plt, joinpath(path, "OU_sim_$(digamma_value)_$(trigamma_value)_$(NUMBER_SEED)_theta.pdf"))
savefig(hipster_tree, joinpath(path, "OU_sim_$(digamma_value)_$(trigamma_value)_hipster_$(NUMBER_SEED)_theta.pdf"))

# Save a representative tree from the MCMC samples as Newick.
Phylo.write(
    joinpath(path, "OU_sim_$(digamma_value)_$(trigamma_value)_HIPSTR_$(NUMBER_SEED)_theta.newick"),
    get_phylo_tree(hip),
)

display(combined_plt)


## 8. Credibility plot for the HIPSTR tree

For each internal node we use the posterior **support** as both a colour
(red→blue, low→high) and a marker size, then annotate the value to the
right of the bubble.

In [ ]:
function max_root_distance(node, dist = 0.0)
    if isleafnode(node)
        return dist
    end
    return maximum(max_root_distance(ch, dist + ch.branchlength) for ch in node.children)
end

for n in getnodelist(hip)
    if isleafnode(n)
        n.node_data = Dict("size" => 2.5, "support_plot" => 0.0)
    else
        s = node2support[n]
        n.node_data = Dict("size" => 8 + 18 * sqrt(s), "support_plot" => s)
    end
end

ladderize!(hip)
phylo_tree = get_phylo_tree(hip)

max_depth  = max_root_distance(hip)
xpad_left  = 0.06 * max_depth
xpad_right = 0.28 * max_depth

pl_cred = plot(
    phylo_tree;
    showtips = true,
    tipfont = 10,
    markersize  = values_from_phylo_tree(phylo_tree, "size"),
    marker_z    = values_from_phylo_tree(phylo_tree, "support_plot"),
    markercolor = cgrad(reverse(ColorSchemes.RdBu_10.colors)),
    clims = (0.0, 1.0),
    markerstrokecolor = :black,
    markerstrokewidth = 0.25,
    linewidth = 1.0,
    xlims = (-xpad_left, max_depth + xpad_right),
    left_margin = 10mm, right_margin = 35mm,
    top_margin = 8mm,   bottom_margin = 8mm,
    colorbar = :right,
    size = (950, 700),
)

# Annotate internal-node support values
scatter_series = pl_cred.series_list[2]
xs = scatter_series[:x]
ys = scatter_series[:y]
zs = scatter_series[:marker_z]
for i in eachindex(xs)
    s = zs[i]
    if s > 0.0       # skip leaves (we set their support to 0)
        annotate!(
            pl_cred,
            xs[i] + 0.07 * max_depth, ys[i],
            text(string(round(s; digits = 3)), 8, :left, :black),
        )
    end
end

savefig(pl_cred, joinpath(path, "OU_sim_$(digamma_value)_$(trigamma_value)_HIPSTR_with_CRED_seed_$(NUMBER_SEED).pdf"))
display(pl_cred)


## 9. Build the clone × node frequency matrix

`run_ou_and_build_clone_matrix` consumes the posterior trees + sampled OU
parameters, recomputes node marginals on the HIPSTR tree, and writes a CSV
of *node × clone* expected frequencies.

In [ ]:
run_ou_and_build_clone_matrix(
    trees,
    models,
    count_matrix,
    cluster_names,
    digamma_value,
    trigamma_value,
    joinpath(path, "OU_sim_$(digamma_value)_$(trigamma_value)_clone_matrix_seed_$(NUMBER_SEED).csv"),
)


## 10. UMAP overlay — load coordinates & build the scatter base

In [ ]:
UMAP_coords = CSV.read(
    joinpath(data_dir, "simulated_UMAP_coords.tsv"),
    DataFrame;
    delim = '\t',
)
UMAP_coords.UMAP_1_new = UMAP_coords.UMAP_1 .* (-1);

UMAP_coords_gdf = groupby(UMAP_coords, :cell_types);
centroids_umap = combine(
    UMAP_coords_gdf,
    [:UMAP_1_new, :UMAP_2] => ((u1, u2) -> (C1 = median(u1), C2 = median(u2))) => AsTable,
);

# Same palette skeleton as the paper's analysis (extra labels that don't
# appear in the simulated data are simply ignored at plot time).
color_palette = Dict(
    "Th17"          => "#82B010",
    "Act2"          => "#8A4AFA",
    "Act3"          => "#4A92FA",
    "Naive"         => "#49ce50",
    "Act_circ"      => "#E5D156",
    "Th_CTL_like"   => "#CE005B",
    "Tfh"           => "#0e7a97",
    "Tregs"         => "#EAA930",
    "Th2"           => "#664382",
    "IFN_response"  => "#8c918c",
    "Proliferating" => "#ff3c00",
)

base_umap_scatter() = scatter(
    UMAP_coords.UMAP_1_new,
    UMAP_coords.UMAP_2;
    group  = UMAP_coords.cell_types,
    color  = [color_palette[c] for c in UMAP_coords.cell_types],
    grid = false, legend = :outerright,
    size = (700, 400), markersize = 3, markerstrokewidth = 0, alpha = 0.4,
)

display(base_umap_scatter())


## 11. Project posterior trees onto the UMAP

For each tree in the posterior:

1.  Initialise an internal Gaussian message template,
2.  Pin every leaf to its cell-type centroid in UMAP space,
3.  Run **Felsenstein** under a 2-D Brownian motion model,
4.  Draw the resulting branches (as faint shadows) on top of the scatter.

After the cloud, we draw the HIPSTR tree on top in solid black.

In [ ]:
sc_p = base_umap_scatter()

for tree in trees
    newtree_umap = copy_tree(tree)
    message_template_umap = [GaussianPartition(0.0, 1000.0), GaussianPartition(0.0, 1000.0)]

    bm_model = [MolecularEvolution.BrownianMotion(0.0, 0.1),
                MolecularEvolution.BrownianMotion(0.0, 0.1)]
    internal_message_init!(newtree_umap, message_template_umap)

    for x in getleaflist(newtree_umap)
        tmp = filter(row -> row[:cell_types] == x.name, centroids_umap)
        x.message = [GaussianPartition(tmp.C1[1], 0.01), GaussianPartition(tmp.C2[1], 0.01)]
    end

    felsenstein!(newtree_umap, bm_model)
    _, sc_p = PlotTreeOnUmapNoAnimShadow(newtree_umap, bm_model, sc_p)
end

# Now overlay the HIPSTR tree on top of the cloud
newtree_umap = copy_tree(hip)
message_template_umap = [GaussianPartition(0.0, 1000.0), GaussianPartition(0.0, 1000.0)]
bm_model = [MolecularEvolution.BrownianMotion(0.0, 0.1),
            MolecularEvolution.BrownianMotion(0.0, 0.1)]
internal_message_init!(newtree_umap, message_template_umap)

for x in getleaflist(newtree_umap)
    tmp = filter(row -> row[:cell_types] == x.name, centroids_umap)
    x.message = [GaussianPartition(tmp.C1[1], 0.01), GaussianPartition(tmp.C2[1], 0.01)]
end
felsenstein!(newtree_umap, bm_model)

_, im_with_hip = PlotTreeOnUmapNoAnim(newtree_umap, bm_model, sc_p);
savefig(im_with_hip, joinpath(path, "sim_with_MCMC_trees.pdf"))
display(im_with_hip)


## 12. HIPSTR-only UMAP overlay (no posterior cloud)

In [ ]:
sc_p = base_umap_scatter()

newtree_umap = copy_tree(hip)
message_template_umap = [GaussianPartition(0.0, 1000.0), GaussianPartition(0.0, 1000.0)]
bm_model = [MolecularEvolution.BrownianMotion(0.0, 0.1),
            MolecularEvolution.BrownianMotion(0.0, 0.1)]
internal_message_init!(newtree_umap, message_template_umap)

for x in getleaflist(newtree_umap)
    tmp = filter(row -> row[:cell_types] == x.name, centroids_umap)
    x.message = [GaussianPartition(tmp.C1[1], 0.01), GaussianPartition(tmp.C2[1], 0.01)]
end
felsenstein!(newtree_umap, bm_model)

_, im_no_cloud = PlotTreeOnUmapNoAnim(newtree_umap, bm_model, sc_p);
savefig(im_no_cloud, joinpath(path, "sim_no_MCMC_trees.pdf"))
display(im_no_cloud)


## 13. Animated GIF of the HIPSTR walk on the UMAP

In [ ]:
sc_p = base_umap_scatter()

newtree_umap = copy_tree(hip)
message_template_umap = [GaussianPartition(0.0, 1000.0), GaussianPartition(0.0, 1000.0)]
bm_model = [MolecularEvolution.BrownianMotion(0.0, 0.5),
            MolecularEvolution.BrownianMotion(0.0, 0.5)]
internal_message_init!(newtree_umap, message_template_umap)

for x in getleaflist(newtree_umap)
    tmp = filter(row -> row[:cell_types] == x.name, centroids_umap)
    x.message = [GaussianPartition(tmp.C1[1], 0.0), GaussianPartition(tmp.C2[1], 0.0)]
end
felsenstein!(newtree_umap, bm_model)

_, im_with_hip_anim = PlotTreeOnUmap(newtree_umap, bm_model, sc_p);
gif(im_with_hip_anim, joinpath(path, "sim_HIPSTR.gif"); fps = 0.5)


## What's in `examples/results/` once the notebook is done

```
OU_sim_<d>_<g>_<seed>_theta.pdf                  # 6-panel diagnostic dashboard
OU_sim_<d>_<g>_hipster_<seed>_theta.pdf          # MCMC cloud + HIPSTR tree
OU_sim_<d>_<g>_HIPSTR_<seed>_theta.newick        # HIPSTR consensus tree (Newick)
OU_sim_<d>_<g>_HIPSTR_with_CRED_seed_<seed>.pdf  # HIPSTR with annotated support
OU_sim_<d>_<g>_clone_matrix_seed_<seed>.csv      # per-clone × per-node frequencies
sim_with_MCMC_trees.pdf                          # UMAP + posterior cloud + HIPSTR
sim_no_MCMC_trees.pdf                            # UMAP + HIPSTR only
sim_HIPSTR.gif                                   # animated walk of HIPSTR on UMAP
```